<h1>Light Spring - Full code</h1>

<h1> Libraries</h1>

In [ ]:
#Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# Motor Libraries
import pathlib, os, time
import libximc.highlevel as ximc
from libximc.highlevel._structure_types import home_settings_t

# Camera Libraries
#import ctypes
# import ft232
import instrumental
import pylablib as pll
pll.par["devices/dlls/thorlabs_tlcam"] = "C:/Program Files/Thorlabs/Scientific Imaging/ThorCam/"
from pylablib.devices import Thorlabs


from pathlib import Path
import os
import re

#Automated data scan
from src.SLM_Lib.src.Hologram import Hologram
from src.SLM_Lib.src.SLMDisplay import SLMScreen, screeninfo
from src.SLM_Lib.src.DataProcessing import *
from src.SLM_Lib.src.Visualizing import *
from imageio import imread
from time import sleep

#OGV
from scipy.optimize import minimize
from math import factorial

<h1>Motor and Camera Setup</h1>

In [ ]:
# ============================================================================================
# Set up the Motor
# Devices search
devices = ximc.enumerate_devices(ximc.EnumerateFlags.ENUMERATE_NETWORK | ximc.EnumerateFlags.ENUMERATE_PROBE)
if len(devices) == 0:
    print("The real devices were not found. A virtual device will be used.")
else:
    # Print real devices list
    print("Found {} real device(s):".format(len(devices)))
    for device in devices:
        print("  {}".format(device))
# Set up the correct one
device_uri = r"xi-com:\\.\COM7" # this is for devices connectd via USB (our case)
axis = ximc.Axis(device_uri)
axis.open_device() # To open the connection, you must manually call `open_device()` method

MIN_MOTOR_POS = -50000
MAX_MOTOR_POS = 50000
MOTOR_RANGE = 5000000 # nm

print( "Initial position:", axis.get_position().Position )
print( "Minimum position:", MIN_MOTOR_POS )
print( "Maximum position:", MAX_MOTOR_POS )
print( "Motor range:", MOTOR_RANGE, "nm" )
print( "Motor step:", MOTOR_RANGE / (MAX_MOTOR_POS-MIN_MOTOR_POS), "nm" )

def get_motor_pos():
    return axis.get_position().Position

def motor_to_origin():
    axis.command_move( 0, 0 )
    axis.command_wait_for_stop(100)

def motor_moveto( pos ):
    global MAX_MOTOR_POS
    global MIN_MOTOR_POS
    print(f"range: [{MIN_MOTOR_POS}; {MAX_MOTOR_POS}] current pos: {axis.get_position().Position}")
    if pos > MAX_MOTOR_POS:
        axis.command_move( MAX_MOTOR_POS, 0 )
        print( "Motor at maximum!" )
    elif pos < MIN_MOTOR_POS:
        axis.command_move( MIN_MOTOR_POS, 0 )
        print( "Motor at minimum!" )
    else:
        axis.command_move( pos, 0 )
    axis.command_wait_for_stop(100)

def motor_step( step ):
    pos = axis.get_position().Position
    if pos + step > MAX_MOTOR_POS:
        axis.command_move( MAX_MOTOR_POS, 0 )
        print( "Motor at maximum!" )
    elif pos + step < MIN_MOTOR_POS:
        axis.command_move( MIN_MOTOR_POS, 0 )
        print( "Motor at minimum!" )
    else:
        axis.command_move( pos + step, 0 )
    axis.command_wait_for_stop(100)

def motor_home(reset_max_pos:bool = True):
    """
    Homes motor by checking for right-sided stop button and setting the position to 0.
    The driving direction is negative afterwards!

    :param reset_max_pos: reset motor max positions to -150000, 0
    :type reset_max_pos: bool
    """
    home_settings = home_settings_t(
        FastHome=10000,
        uFastHome=0,
        SlowHome=2000,
        uSlowHome=0,
        HomeDelta=-5,
        uHomeDelta=0,
        #maybe still adjustable for better settings?
        HomeFlags=ximc.HomeFlags.HOME_DIR_FIRST|ximc.HomeFlags.HOME_STOP_FIRST_LIM
    )
    global MIN_MOTOR_POS
    global MAX_MOTOR_POS
    print("Motor is homing...")
    axis.set_home_settings(home_settings)
    axis.command_home()
    axis.command_wait_for_stop(100)
    axis.command_zero()
    if reset_max_pos:
        MAX_MOTOR_POS=0
        MIN_MOTOR_POS=-150000
    print("Finished homing")

# Set up the Camera
cameras_SN_list = Thorlabs.list_cameras_tlcam()
print("Camera list = " + str(cameras_SN_list)) 
cam_selected = 17943
print( "Selected Camera: ", cam_selected )
try: 
    ccd.close()
except: 
    pass
ccd = Thorlabs.ThorlabsTLCamera( cam_selected )
# Set up the exposure
EXPOSURE = 1
ccd.set_exposure( EXPOSURE )
print( "Exposure set to ", EXPOSURE )
PIXEL_SIZE = 3.3 # micron ?

######################################################################################################
# Funtions for image processing
def take_picture(exposure:float, remove_dead_pixels = False):
    ccd.set_exposure(exposure)
    img = np.asarray(ccd.snap())

    if remove_dead_pixels:
        dead_pixels = img / img.max() > 0.95
        img[ dead_pixels ] = 0
    return img

def crop_defined(image:np.ndarray,center:tuple[int,int], abs_x:int, abs_y:int)->np.ndarray:
    """ 
    Crops the given image to a rectangular geometrz defined by the following parameters:

    :param image: Image data to be cropped
    :type image: numpy.ndarray

    :param center: Center coordinate of the rectangle
    :type center: tuple[int,int]

    :param abs_x: absolute width in x direction (horizontal)
    :type abs_x: int

    :param abs_y: absolute hight in y direction (vertically)
    :type abs_y: int
    """
    min_x = center[0] - int(abs_x/2)
    max_x = center[0] + int(abs_x/2)
    min_y = center[1] - int(abs_y/2)
    max_y = center[1] + int(abs_y/2)
    if min_x < 0:
        min_x = 0
    if max_x > image.shape[0]:
        max_x = image.shape[0]
    if min_y < 0:
        min_y = 0
    if max_y > image.shape[1]:
        max_y = image.shape[1]
    newImage = image[min_x:max_x, min_y:max_y]
    return newImage

def analyseFFT(image:np.ndarray, kmin:float, plot:bool = True): #Add to the lib
    nx, ny = image.shape
    kx = np.fft.fftshift( np.fft.fftfreq( nx, PIXEL_SIZE ) ) * 2*np.pi
    ky = np.fft.fftshift( np.fft.fftfreq( ny, PIXEL_SIZE ) ) * 2*np.pi
    KX, KY = np.meshgrid( ky, kx )
    # fourier transform
    fft = np.fft.fftshift(np.fft.fft2(image))
    fft /= np.max( np.abs( fft ) )
    # get intensity maximum of higher frequency components
    intensity = np.max( np.abs( fft[ np.sqrt(KX**2+KY**2) > kmin ] ) )
    
    if plot:
        plt.close("all")
        #print('plotting')
        fft[ np.sqrt(KX**2+KY**2) < kmin ] = 0
        
        fig, [ax1, ax2] = plt.subplots( 1, 2, figsize=[10,5] )
        
        plt.sca(ax1)
        plt.imshow( image,
                    origin="lower",
                    cmap="rainbow" )
        plt.colorbar()
        #plt.xlim( 1000, 2000 )
        #plt.ylim( 2500, 2500 )
        plt.title( "CCD" )
        
        plt.sca(ax2)
        plt.imshow( np.abs( fft ),
                    extent=[ np.min(kx), np.max(kx), np.min(ky), np.max(ky) ],
                    origin="lower",
                    cmap="hot",
                    norm=mpl.colors.LogNorm( vmin=1e-5 ))
        c1 = plt.Circle((0,0),kmin,color = "green")
        ax2.add_patch(c1)
        plt.colorbar()
        plt.title( "Absolute Value of FFT" )
        
        plt.suptitle( "Pos = "+ str( get_motor_pos() ) )
        plt.show()

    return( get_motor_pos(),intensity )

def get_object_position( obj, plot=False, rmv_dead_pixels=False ):

    if rmv_dead_pixels:
        dead_pixels = obj / obj.max() > 0.95
        obj[ dead_pixels ] = 0

    nx, ny = obj.shape
    obj_x = int( np.average( np.arange(nx), weights=np.sum(obj.T, 0) ) )
    obj_y = int( np.average( np.arange(ny), weights=np.sum(obj.T, 1) ) )
    obj_n = np.min( [ obj_x, obj_y, nx-obj_x, ny-obj_y ] )

    if plot:
        fig, ax = plt.subplots( 1, 2, figsize=[10,5] )

        ax[0].imshow( obj.T, cmap="rainbow", origin="lower", aspect="auto" )
        ax[0].scatter( [obj_x], [obj_y], c="black", marker="x" )
        ax[0].plot( np.sum(obj.T, 0)/np.max(np.sum(obj.T, 0)) * ny/4, c="k", lw=1 )
        ax[0].plot( np.sum(obj.T, 1)/np.max(np.sum(obj.T, 1)) * nx/4, np.arange(ny), c="k", lw=1 )

        ax[0].axvline( obj_x, c="r", lw=1, ls="--" )
        ax[0].axhline( obj_y, c="r", lw=1, ls="--" )

        ax[1].imshow( obj[obj_x-obj_n:obj_x+obj_n, obj_y-obj_n:obj_y+obj_n].T, cmap="rainbow", origin="lower" )
        ax[1].axvline( obj_n, c="black", ls="--" )
        ax[1].axhline( obj_n, c="black", ls="--" )
        plt.show()

    return obj_x, obj_y, obj_n

In [ ]:
# motor_home()
# print(get_motor_pos())
# motor_moveto(-30000)
# print(get_motor_pos())

print(get_motor_pos())
motor_step(1)
print(get_motor_pos())
motor_step(1)
print(get_motor_pos())
motor_step(1)



<h1> SLM Hologram </h1>

In [ ]:
%matplotlib inline
slm = SLMScreen(1)
h = Hologram()
h.loadPhaseCalibrationFile(Path(r"C:\Users\Voxel\Desktop\Data_analysis_LS\Correction_patterns/CAL_LSH0803840_785nm.bmp"),
                      plot=False)
h.loadVoltageCalibrationFile(Path(r"C:\Users\Voxel\Desktop\LightSprings\polyfit_par.txt"))
h.setZernikeCorrection(verAstic=.03)
h.lightspring(90,1,9,5.63,6.05)
#h.stepFunctionOAM(14, 7, 30, 20, 1, -1)
#h.vortex(90)
h.plotHolo()
h.plotModes()
slm.updateImage(h.convertHolo())
# for i in range(20,100,20):
#     h.lightspring(i,1,9,5.63,6.05)
#     slm.updateImage(h.convertHolo())
#     h.plotHolo()
#     sleep(2)
# slm.close()

<h2>Close Hologram </h2>

In [ ]:
#close the slm Window without crashing the kernel
slm.close()

<h1> Temporal Overlap - FFT </h1>

<h2>Single Picture</h2>

In [ ]:
t = np.load(r"C:\Users\Voxel\Desktop\LightSprings\Data\250520\lightspring_OAM20_SLM=9_n=9_dl=1\lightspring_tc20_SLM=9_n=9_dl=1_motorPos-30242.npy")
plt.figure()
plt.imshow(t)
plt.show()
print(t.shape)

In [ ]:
# single picture
%matplotlib inline
#motor_home()
#-115700, -112800,
#motor_moveto(-112800)
save = 0
savePath = pathlib.Path(r"C:\Users\Voxel\Desktop\LightSprings\Data")/ pathlib.Path(time.strftime("%y%m%d",time.gmtime())) /"correctionData"
if not os.path.exists(savePath):
    os.makedirs(savePath)
angle = 1
beams = ["Ref", "Main","Interf"]
beam = 2
pos = get_motor_pos()
filename = f"LightSpring_Filter_7OAM_{angle}_{beams[beam]}"
filename = f"Lightspring_80_86OAM_centered_{beams[beam]}_2"
filename = f"Vortex_70OAM_pos{pos}_{beams[beam]}"
filename = f"correctionWave_{beams[beam]}_{pos}"

fig = plt.figure()
#while True:
img = take_picture(0.001, 1)
#img = np.load(r"C:\Users\Voxel\Desktop\LightSprings\Data\19-03-25\OAM-Anglesfilter\0_deg.npy")
#print(img.shape)
#print(img)
cropped = img#crop_defined(img,(500,700), 650, 650)# with grating: crop_defined(img,(250,600), 450, 450)
#print(cropped.shape)
#print(cropped)
ax = plt.gca()
ax.imshow(cropped, cmap="rainbow")
analyseFFT(cropped, kmin=0.0)
if save:
    np.savetxt((savePath/filename).with_suffix(".csv"), img, delimiter=";")
    np.save(savePath/filename, img)

<h2> Full Temporal Overlap Scan</h2>

In [ ]:
#setup
%matplotlib inline
remove_dead_pixels= 1
ccd.set_exposure(0.001)
pos_min, pos_max, step = -25000, -30000, 100
kmin = 0.06
    # crop
do_crop = 0
c_center, c_x, c_y = (600,1150), 250, 250

plot = True
save = False
OAM_charge = 10
filenames = "C:/Users/Voxel/Desktop/LightSprings/Data/04-07-24/LS_10_8_8_6_6_4_4_2_2_0_0/"
####
data_pos = []
data_int = []
motor_home()
motor_moveto( pos_min )
ccd.start_acquisition( nframes=1 )
while get_motor_pos() < pos_max:  
    # get data from camera
    ccd.wait_for_frame()
    img = ccd.read_oldest_image()
    if remove_dead_pixels:
        dead_pixels = img / img.max() > 0.95
        img[ dead_pixels ] = 0
    # fourier transform
    if do_crop:
        cropped = crop_defined(img, c_center, c_x, c_y)
    else:
        cropped = img

    res = analyseFFT(cropped, kmin, plot)
    
    data_pos.append(res[0])
    data_int.append(res[1])

    # save
    if save:
        img_save = crop_img( img )
        if not os.path.exists(filenames+ dataset):
            os.mkdir(filenames+ dataset)
        np.save( filenames+ dataset +"motor%06i"%( get_motor_pos() - MIN_MOTOR_POS ), img_save )

    print( data_pos[-1], data_int[-1], "       ", end="\r" )
    
    motor_step(step)
ccd.stop_acquisition()

<h2> Plot Temporal Overlap Scan </h2>

In [ ]:
xmin, xmax = pos_min, pos_max
%matplotlib tk
fig = plt.figure()
plt.scatter( data_pos, data_int, s=1 )
plt.axvline(xmin, lw=1, c="r", ls="--")
plt.axvline(xmax, lw=1, c="r", ls="--")
#plt.xlim( -11000, -4000 )
plt.xlabel( "Motor Position" )
plt.ylabel( "Maximum off axis FFT component" )
plt.show()

fig = plt.figure()
plt.scatter( data_pos, data_int, s=1 )
plt.xlim( xmin, xmax )
plt.xlabel( "Motor Position" )
plt.ylabel( "Maximum off axis FFT component" )
plt.grid()
plt.show()


<h1>Data Acquisition</h1>

<h2> Automated scan to aquire data </h2>

In [ ]:
%matplotlib inline
# aquisition
show = 0
remove_dead_pixels= 1
ccd.set_exposure(0.001)
pos_min, pos_max, step = -32500, -32400, 1 #-33200, -30220, 25
# path and names
path = pathlib.Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\\"+time.strftime("%y%m%d",time.gmtime()))
structureName = "Test"
topologicalCharge = ""
additionalName = ""
subfolder = pathlib.Path(f"{structureName}_tc{topologicalCharge}_{additionalName}")
#
####
stem = structureName+f"_tc{topologicalCharge}"+f"_{additionalName}"
data_pos = []
data_int = []
motor_home()
motor_moveto( pos_min )
ccd.start_acquisition( nframes=1 )
n_images = np.abs(pos_max - pos_min) // step
print(f"Starting to aquire images:\n  savepath = {path}\n  stemname = {stem}\n  n_images = {n_images}")
i = 1
if not subfolder.__str__ == "":
    path = path/subfolder
while (get_motor_pos() < pos_max):  
    # get data from camera
    ccd.wait_for_frame()
    img = ccd.read_oldest_image()
    if remove_dead_pixels:
        dead_pixels = img / img.max() > 0.95
        img[ dead_pixels ] = 0
    # # fourier transform
    # if do_crop:
    #     cropped = crop_defined(img, c_center, c_x, c_y)
    # else:
    #     cropped = img

    # res = analyseFFT(cropped, kmin, plot)
    # data_pos.append(res[0])
    # data_int.append(res[1])
    if show:
        fig = plt.figure()
        a = plt.imshow(img, cmap = "inferno")
        plt.colorbar(a)
        plt.show()
    # save
    if not os.path.exists(path):
        os.makedirs(path)
    np.save(path/pathlib.Path(stem+"_motorPos%i"%( get_motor_pos())),
            img)
    print(f"aquired image {i}/{n_images}")
    i = i +1
    motor_step(step)
ccd.stop_acquisition()
print("Finished")

<h2>Multi Parameter Measurement sets </h2>

In [ ]:
%matplotlib inline
holo = Hologram()

SCREENID = 1
SLM = SLMScreen(SCREENID)
holo.loadPhaseCalibrationFile(
    Path(r"C:\Users\Voxel\Desktop\Data_analysis_LS\Correction_patterns/CAL_LSH0803840_785nm.bmp")
)
holo.loadVoltageCalibrationFile(
    Path(r"C:\Users\Voxel\Desktop\LightSprings\polyfit_par.txt")
)
holo.setZernikeCorrection(verAstic=0.03)
holo.lightspring(30, 1, 9, 5.63, 6.05)
SLM.updateImage(holo.convertHolo())

In [ ]:

#keys
N_FREQUENCYBINS = "n"
TOPOLOGICALCHARGE = "OAM"
SLM_POSITION = "SLM"
DELTA_L = "dl"
TOTAL_DL = "totaldl"

parameterSet = {
    SLM_POSITION:[9],
    N_FREQUENCYBINS:[9], #np.linspace(3,19,17, dtype=int),
    TOPOLOGICALCHARGE:[20, 25, 60, 100, 120],
    DELTA_L:[1],
    TOTAL_DL:None #np.linspace(6,60,10, dtype=int)
}

def buildHologram(parameters:dict, rMin, rMax, holo:Hologram = None, plot = False):
    if holo is None: holo = Hologram()
    holo.resetHolo()
    holo.lightspring(
        OAM=parameters[TOPOLOGICALCHARGE],
        deltaL=parameters[DELTA_L],
        nSample=parameters[N_FREQUENCYBINS],
        rMin=rMin, rMax=rMax
    )
    # centered OAM
    # OAMs = np.ones((parameters[N_FREQUENCYBINS]))*parameters[TOPOLOGICALCHARGE]
    # if parameters[N_FREQUENCYBINS] % 2 == 0:
    #     OAMs = np.linspace(parameters[TOPOLOGICALCHARGE]-np.floor(parameters[N_FREQUENCYBINS]/2),
    #                             parameters[TOPOLOGICALCHARGE]+np.floor(parameters[N_FREQUENCYBINS]/2)-1,parameters[N_FREQUENCYBINS])
    # else:
    #     OAMs = np.linspace(parameters[TOPOLOGICALCHARGE]-np.floor(parameters[N_FREQUENCYBINS]/2),
    #                             parameters[TOPOLOGICALCHARGE]+np.floor(parameters[N_FREQUENCYBINS]/2),parameters[N_FREQUENCYBINS])
    # radia = holo._linearInterpolRadius(parameters[N_FREQUENCYBINS],10)[0]
    # holo.customFunction(radia, OAMs)
    # print(f"OAMs = {OAMs}")
    if plot:
        holo.plotHolo()
        holo.plotModes()
    
    # ## centered OAM with fixed total DeltaL
    # OAMs = np.ones((parameters[N_FREQUENCYBINS]))*parameters[TOPOLOGICALCHARGE]
    # dl = parameters[TOTAL_DL] / parameters[N_FREQUENCYBINS]
    # if parameters[N_FREQUENCYBINS] % 2 == 0:
    #     OAMs = OAMs + np.round(np.linspace(-np.floor(parameters[N_FREQUENCYBINS]/2),
    #                         np.floor(parameters[N_FREQUENCYBINS]/2)-1,
    #                         parameters[N_FREQUENCYBINS])*dl,0)
    # else:
    #     OAMs = OAMs + np.round(np.linspace(-np.floor(parameters[N_FREQUENCYBINS]/2),
    #                         np.floor(parameters[N_FREQUENCYBINS]/2),
    #                         parameters[N_FREQUENCYBINS])*dl,0)
        
    # radia = holo._linearInterpolRadius(parameters[N_FREQUENCYBINS],10)[0]
    # holo.customFunction(radia, OAMs)
    # print(f"OAMs = {OAMs}")
    # if plot:
    #     holo.plotHolo()
    #     holo.plotModes()

def scanForSLMPos(slmPos, structureName, rMin, rMax,
                  pos_min, pos_max, step):
    holo.rMin, holo.rMax = rMin, rMax
    for n in parameterSet[N_FREQUENCYBINS]:
        for OAM in parameterSet[TOPOLOGICALCHARGE]:
            if not parameterSet[DELTA_L] is None:
                for dl in parameterSet[DELTA_L]:
                    # buildHologram({SLM_POSITION:slmPos,
                    #      N_FREQUENCYBINS:n,
                    #      TOPOLOGICALCHARGE:OAM,
                    #      DELTA_L:dl},rMin,rMax,holo)
                    # sleep(2)
                    scanSingle(
                        {SLM_POSITION:slmPos,
                        N_FREQUENCYBINS:n,
                        TOPOLOGICALCHARGE:OAM,
                        DELTA_L:dl},
                        structureName, rMin,rMax,
                        pos_min, pos_max, step
                    )
            elif not parameterSet[TOTAL_DL] is None:
                for tdl in parameterSet[TOTAL_DL]:
                    scanSingle(
                        {SLM_POSITION:slmPos,
                        N_FREQUENCYBINS:n,
                        TOPOLOGICALCHARGE:OAM,
                        TOTAL_DL:tdl},
                        structureName, rMin,rMax,
                        pos_min, pos_max, step
                    )

def scanSingle(parameters:dict, structureName, rMin,rMax,
               pos_min, pos_max, step,
               show = 0, remove_dead_pixels = 1):
    #build hologram
    buildHologram(parameters,rMin,rMax,holo, True)
    SLM.updateImage(holo.convertHolo())
    # aquisition
    ccd.set_exposure(0.001)
    # path and names from sructure
    path = pathlib.Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\\"+time.strftime("%y%m%d",time.gmtime()))
    structureName = structureName
    topologicalCharge = parameters[TOPOLOGICALCHARGE]
    additionalName = "_".join([f"{key}={val}" for key, val in parameters.items() if not (key == TOPOLOGICALCHARGE)])
    subfolder = pathlib.Path(f"{structureName}_OAM{topologicalCharge}_{additionalName}")
    #
    ####
    stem = structureName+f"_tc{topologicalCharge}"+f"_{additionalName}"
    motor_home()
    motor_moveto( pos_min )
    ccd.start_acquisition( nframes=1 )
    n_images = np.abs(pos_max - pos_min) // step
    print(f"Starting to aquire images:\nParameters = {parameters}\n  savepath = {path}\n  stemname = {stem}\n  n_images = {n_images}")
    i = 1
    if not subfolder.__str__ == "":
        path = path/subfolder
    while (get_motor_pos() < pos_max):  
        # get data from camera
        ccd.wait_for_frame()
        img = ccd.read_oldest_image()
        if remove_dead_pixels:
            dead_pixels = img / img.max() > 0.95
            img[ dead_pixels ] = 0

        if show:
            fig = plt.figure()
            a = plt.imshow(img, cmap = "inferno")
            plt.colorbar(a)
            plt.show()
        # save
        if not os.path.exists(path):
            os.makedirs(path)
        np.save(path/pathlib.Path(stem+"_motorPos%i"%( get_motor_pos())),
                img)
        print(f"aquired image {i}/{n_images}")
        i = i +1
        motor_step(step)
    ccd.stop_acquisition()
    print("Finished Scan")

scanForSLMPos(parameterSet[SLM_POSITION][0], "lightspring", 5.63, 6.05, -33200, -30220, 25)

<h1>Data Processing</h1>

<h2>Single Reconstruction</h2>

In [ ]:
%matplotlib inline
rotangle = 2.5
timing = 6201
# cx=437
# cy=359
# rin=33
# rout=45
ellipse = (
    503,    #x
    504,    #y
    102,     #A horizontal
    104,     #B vertical
    10,     #thickness
    0       #angle
)
path = Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250520\lightspring_OAM20_SLM=9_n=9_dl=1")
n = len(os.listdir(path))
crop_pos = ((600,500),1000)
shift = (np.around(np.linspace(0,3,n)), np.zeros(n, dtype=int))

#crop_pos = crop_pos = ((550,660),700)
#path = Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250520")
#file_base = Path("lightspring_OAM20_SLM=9_n=9_dl=1\lightspring_tc20_SLM=9_n=9_dl=1_motorPos-30242.npy")
file = r"lightspring_tc60_SLM=9_n=9_dl=1_motorPos-31952.npy"
#reconstructData( path/file, cx, cy, rin, rout, crop_pos, plot_all=True, plot_reconstruction=True,fft_crop_radius=6,circle_shift=(0,0) )
reconstructData( path/file, *ellipse, crop_pos, plot_all = True,
                      fft_crop_radius=5.5)

<h2>Reconstruction from Folders</h2>

In [ ]:
%matplotlib inline
# 30 till 90 OAM
# cx=436
# cy=359
# rin=33 
# rout=44
# crop_pos = crop_pos = ((550,660),1000)
# shift = (np.linspace(0,3,28, dtype=int), np.zeros(28, dtype=int))
root = Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250520")
path= root/Path("lightspring_OAM60_SLM=9_n=9_dl=1")
#fileName = r"C:\Users\Voxel\Desktop\LightSprings\Data\250513" #Single File
ellipse = (
    503,    #x
    504,    #y
    102,     #A horizontal
    104,     #B vertical
    10,     #thickness
    0       #angle
)
n = len(os.listdir(path))
crop_pos = ((600,500),1000)
shift = (np.around(np.linspace(0,3,n)), np.zeros(n, dtype=int))

print(shift)
reconstructFromFolder(path, *ellipse, crop_pos, removeWrongExposure=True, phaselock=0, shift=shift,
                      plot_exp_hist=True, plot_all = True,
                      fft_crop_radius=5.5)#, subtract_ref=root/Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250512\lighspring_OAM50_n=9_sligthBiggerAngle\correctionData\correctionWave_Interf_-112800.npy"))
# Tests phase tilt
#reconstructFromFolder(path, cx, cy, rin, rout, crop_pos, tilt_correction=1, phaselock=0, plot_all=0, shift=shift)

<h2>Manual FFT Reconstruction</h2>

In [ ]:
# from nfft import nfft
# from BPF_calib import lambda_f
# lambdas = lambda_f(rotangles)
# print(lambdas)
# FIELD = np.fft.fft(hypervolume)
%matplotlib inline
# FIELD = np.fft.fft(phaselockField2(hypervolume))
print(omegas)
FIELD = manualFFT(hypervolume, omegas, 2e-15)
print(FIELD)
plt.figure()
plt.imshow(np.abs(FIELD),cmap="inferno",vmin=0)
plt.colorbar()
plt.title("reconstructed field")
print(FIELD.shape)

<h2>Animation</h2>

In [ ]:
%matplotlib tk
# animation
from matplotlib.animation import FuncAnimation, PillowWriter

fig, ax = plt.subplots()
i = 0,
frames = 200
t = np.linspace(-200e-15,200e-15,frames)
ln = ax.imshow(np.abs(manualFFT(hypervolume, omegas, t[0])), cmap = "inferno", vmin = 0)
cbar = fig.colorbar(ln, cmap = "inferno")
def init():
    print(i)
    return [ln]

def update(frame):
    ax.set_title(f"[{t[0]*1e15}; {t[-1]*1e15}]fs; {frames} frames;   t = {round(t[frame]*1e15,2)}fs")
    ln.set_data(np.abs(manualFFT(hypervolume, omegas, t[frame], 1)))
    cbar.update_normal(ln)
    return [ln]

ani = FuncAnimation(fig, update, frames = frames, blit= True)
# To save the animation using Pillow as a gif
writer = PillowWriter(fps=15,
           metadata=dict(artist='Me'),
           bitrate=1800)
ani.save(Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\31-03-25")/"lightspring_phaselock_subtraction.gif", writer=writer)

<h2> Plot 3D Isosurface </h2>

In [ ]:
root = Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250513")
path= root/Path("lightSpring_OAM30_smallAngle_n=9")
data = makeVolumeArray(path)
plotReconstruction3D(data, surfaceCount=1, minRel=0.4, maxRel=0.4, downscale=1, opacity=.5)

<h1>OGV Analysis</h1>

<h2>OAM=90</h2>

In [ ]:
%matplotlib inline
path_root = Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250519\LS_tc90_dl=1\reconstructionData")
fname = Path("LS_tc30__motorPos-30672_reconstruction_complex.npy")
#paths = [r"C:\Users\Voxel\Desktop\LightSprings\Data\250411\OAM50to56dl1Check\Lightspring_tc50to56_dl1Check_motorPos-62077.npy",
 #       r"C:\Users\Voxel\Desktop\LightSprings\Data\250411\OAM50to56dl1Check\Lightspring_tc50to56_dl1Check_motorPos-62584.npy",
  #      r"C:\Users\Voxel\Desktop\LightSprings\Data\250411\OAM80to86dl1Check\Lightspring_tc80to86_dl1Check_motorPos-61730.npy",
   #     r"C:\Users\Voxel\Desktop\LightSprings\Data\250411\OAM80to86dl1Check\Lightspring_tc80to86_dl1Check_motorPos-62469.npy",
    #    r"C:\Users\Voxel\Desktop\LightSprings\Data\250411\OAM120to126dl1Check\Lightspring_tc120to126_dl1Check_motorPos-61864.npy",
     #   r"C:\Users\Voxel\Desktop\LightSprings\Data\250411\OAM120to126dl1Check\Lightspring_tc120to126_dl1Check_motorPos-62235.npy",
      #  r"C:\Users\Voxel\Desktop\LightSprings\Data\250417\lightspring_OAM18_dl1\lightspring_tc18_dl1_motorPos-62481.npy",
       # r"C:\Users\Voxel\Desktop\LightSprings\Data\250417\lightspring_OAM18_dl1\lightspring_tc18_dl1_motorPos-62787.npy",
       # r"C:\Users\Voxel\Desktop\LightSprings\Data\250417\lightspring_OAM25_dl-1\lightspring_tc25_dl-1_motorPos-62183.npy",
       # r"C:\Users\Voxel\Desktop\LightSprings\Data\250417\lightspring_OAM25_dl-1\lightspring_tc25_dl-1_motorPos-62721.npy",
       # r"C:\Users\Voxel\Desktop\LightSprings\Data\250417\lightspring_OAM30_dl1\lightspring_tc30_dl1_motorPos-62129.npy",
       # r"C:\Users\Voxel\Desktop\LightSprings\Data\250417\lightspring_OAM30_dl1\lightspring_tc30_dl1_motorPos-62619.npy"
#]
# for file in paths:
# img = np.load(Path(path_root/fname))
# angle = getCurrentAngleLS(img, threshhold=0.8, center=(345,320), maskRadius=20, showPlot=1, showFit=1)
# print(angle)
data90, timestamps90 = makeVolumeArray(path_root)
center=(345,320)
maskRadius=40
threshhold=0.8
print("weighted 90")
ogv_data_weighted90 = getOGV(
    data90,timestamps90,
    center, maskRadius, threshhold,sigma_Fit=0.8,
    showStatistics=0, showFit=True,
    angleFittingAlgo=1, 
    fromIsosurface=0
)


In [ ]:
print(ogv_data_weighted90[4])
plt.figure()
plt.plot(timestamps90, ogv_data_weighted90[4])
plt.show()

<h2>OAM=30</h2>

In [ ]:
%matplotlib inline
data30, timestamps30 = makeVolumeArray(Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250519\LS_tc30_dl=1\reconstructionData"))
center=(345,320)
maskRadius=40
threshhold=0.8
print("weighted 30")
ogv_data_weighted30 = getOGV(
    data30,timestamps30,
    center, maskRadius, threshhold,sigma_Fit=0.8,
    showStatistics=0, showFit=True,
    angleFittingAlgo=1, 
    fromIsosurface=0
)

<h2>OAM=20</h2>

In [ ]:
%matplotlib inline
data20, timestamps20 = makeVolumeArray(Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250520\lightspring_OAM20_SLM=9_n=9_dl=1\reconstructionData"))
center=(345,320)
maskRadius=20
threshhold=0.8
print("weighted 20")
ogv_data_weighted20 = getOGV(
    data20,timestamps20,
    center, maskRadius, threshhold,sigma_Fit=0.8,
    showStatistics=0, showFit=True,
    angleFittingAlgo=1, 
    fromIsosurface=0
)

<h2>OAM=25</h2>

In [ ]:
%matplotlib inline
data25, timestamps25 = makeVolumeArray(Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250520\lightspring_OAM25_SLM=9_n=9_dl=1\reconstructionData"))
center=(345,320)
maskRadius=20
threshhold=0.8
print("weighted 25")
ogv_data_weighted25 = getOGV(
    data25,timestamps25,
    center, maskRadius, threshhold,sigma_Fit=0.8,
    showStatistics=0, showFit=True,
    angleFittingAlgo=1, 
    fromIsosurface=0
)

<h2>OAM=60</H2>

In [ ]:
%matplotlib inline
data60, timestamps60 = makeVolumeArray(Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250520\lightspring_OAM60_SLM=9_n=9_dl=1\reconstructionData"))
center=(345,320)
maskRadius=40
threshhold=0.8
print("weighted 60")
ogv_data_weighted60 = getOGV(
    data60,timestamps60,
    center, maskRadius, threshhold,sigma_Fit=0.8,
    showStatistics=0, showFit=True,
    angleFittingAlgo=1
)

<h2>OAM=100</h2>

In [ ]:
%matplotlib inline
data100, timestamps100 = makeVolumeArray(Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250520\lightspring_OAM100_SLM=9_n=9_dl=1\reconstructionData"))
center=(345,320)
maskRadius=40
threshhold=0.8
print("weighted 100")
ogv_data_weighted100 = getOGV(
    data100,timestamps100,
    center, maskRadius, threshhold,sigma_Fit=0.8,
    showStatistics=0, showFit=True,
    angleFittingAlgo=1
)

<h2>OAM=120</h2>

In [ ]:
%matplotlib inline
data120, timestamps120 = makeVolumeArray(Path(r"C:\Users\Voxel\Desktop\LightSprings\Data\250520\lightspring_OAM120_SLM=9_n=9_dl=1\reconstructionData"))
center=(345,320)
maskRadius=40
threshhold=0.8
print("weighted 120")
ogv_data_weighted120 = getOGV(
    data120,timestamps120,
    center, maskRadius, threshhold,sigma_Fit=0.8,
    showStatistics=0, showFit=True,
    angleFittingAlgo=1
)
# print("maximum")
# ogv_data_max = getOGV(
#     data, np.linspace(0,data.shape[0]-1,data.shape[0]),center, maskRadius, threshhold,sigma_Fit=0.8,
#     showStatistics=True, showFit=True,
#     angleFittingAlgo=0
# )

<h2>Angle Plotting </h2>

In [ ]:
#test isosurface
iso = np.copy(data90)
mask = makeIsosurfaceMask(iso, 0.5, 10)
iso[~mask] = 0
for i in range(0,iso.shape[0],2):
    X, Y = np.meshgrid(np.linspace(0,iso.shape[1]-1, iso.shape[1]), np.linspace(0,iso.shape[2]-1, iso.shape[2]))
    plt.figure()
    plt.imshow(np.abs(iso[i,:,:]), cmap="plasma")
    plt.colorbar()
    
    plt.scatter(X[mask[i,:,:]], Y[mask[i,:,:]], marker="x", color = "y")
    plt.show()

In [ ]:
%matplotlib tk
plt.close("all")
plt.figure()

plt.plot(timestamps20*1e15,ogv_data_weighted20[-1], ls = ":", label = "angle weighted t20")
plt.plot(timestamps25*1e15,ogv_data_weighted25[-1], ls = ":", label = "angle weighted tc25")
plt.plot(timestamps30*1e15,ogv_data_weighted30[-1], ls = ":", label = "angle weighted tc30")
plt.plot(timestamps60*1e15,ogv_data_weighted60[-1], ls = ":", label = "angle weighted tc60")
plt.plot(timestamps90*1e15,ogv_data_weighted90[-1], ls = ":", label = "angle weighted tc90")
plt.plot(timestamps100*1e15,ogv_data_weighted100[-1], ls = ":", label = "angle weighted tc100")
plt.plot(timestamps120*1e15,ogv_data_weighted120[-1], ls = ":", label = "angle weighted tc120")
#plt.plot(timestamps90*1e15,np.gradient(ogv_data_weighted90[-1],timestamps90*1e15))
#plt.plot(timestamps30*1e15,np.gradient(ogv_data_weighted30[-1],timestamps30*1e15))
# plt.plot(timestamps*1e14,ogv_data_max[-1], label = "angle max")
plt.xlabel("time/fs")
plt.ylabel("angles/rad")
plt.legend()
plt.show(
)


In [ ]:
%matplotlib tk
plt.close("all")
# plt.figure()
# plt.plot(timestamps90*1e15,ogv_data_weighted90[-1], ls = ":", label = "angle weighted tc90")
# plt.plot(timestamps30*1e15,ogv_data_weighted30[-1], ls = ":", label = "angle weighted tc30")
# plt.plot(timestamps90*1e15,np.gradient(ogv_data_weighted90[-1],timestamps90*1e15))
# plt.plot(timestamps30*1e15,np.gradient(ogv_data_weighted30[-1],timestamps30*1e15))
# plt.plot(timestamps*1e14,ogv_data_max[-1], label = "angle max")
# plt.xlabel("time/fs")
# plt.ylabel("angles/rad")
# plt.legend()
# plt.show(
# )

# plt.figure()
# plt.plot((timestamps90*1e15),ogv_data_weighted90[2]/C0, label = "OGV_tc90 (dphi/dt)")
# plt.plot((timestamps30*1e15),ogv_data_weighted30[2]/C0, label = "OGV_tc30 (dphi/dt)")
# plt.xlabel("time/fs")
# plt.ylabel("OGV/ c")
# plt.legend()
# plt.show()


# plt.figure()
# plt.plot((timestamps90*1e15),ogv_data_weighted90[3], label = "radius 90")
# plt.plot((timestamps30*1e15),ogv_data_weighted30[3], label = "radius 30")
# plt.plot((timestamps90*1e15),np.gradient(ogv_data_weighted90[3],(timestamps90*1e15)), ls = ":")
# plt.plot((timestamps30*1e15),np.gradient(ogv_data_weighted30[3],(timestamps30*1e15)), ls = "--")
# plt.ylabel("radius/m")
# plt.xlabel("time/fs")
# plt.legend()
# plt.show()

# plt.figure()
# plt.plot(timestamps90*1e15, ogv_data_weighted90[2]*ogv_data_weighted90[3]/C0, label = "LOGV 90")
# plt.plot(timestamps30*1e15, ogv_data_weighted30[2]*ogv_data_weighted30[3]/C0, label = "LOGV 30")
# plt.title("LOGV")
# plt.ylabel("LOGV/ c")
# plt.xlabel("time/fs")
# plt.legend()
# plt.show()

plt.figure()
plt.plot(timestamps20*1e15, np.unwrap(ogv_data_weighted20[-1]), ls=":", marker="*", label="angle weighted tc20", color="r")
plt.plot(timestamps25*1e15, np.unwrap(ogv_data_weighted25[-1]), ls=":", marker="^", label="angle weighted tc25", color="orange")
plt.plot(timestamps30*1e15, np.unwrap(ogv_data_weighted30[-1]), ls=":", marker="<", label="angle weighted tc30", color="y")
plt.plot(timestamps60*1e15, np.unwrap(ogv_data_weighted60[-1]), ls=":", marker=".", label="angle weighted tc60", color="g")
plt.plot(timestamps90*1e15, np.unwrap(ogv_data_weighted90[-1]), ls=":", marker="o", label="angle weighted tc90", color="b")
plt.plot(timestamps100*1e15, np.unwrap(ogv_data_weighted100[-1]), ls=":", marker="v", label="angle weighted tc100", color="pink")
plt.plot(timestamps120*1e15, np.unwrap(ogv_data_weighted120[-1]), ls=":", marker="+", label="angle weighted tc120", color="purple")

plt.xlabel("time/fs")
plt.ylabel("angles/rad")
plt.legend()
plt.show()


ogv_data_weighted20unwrap = np.unwrap(ogv_data_weighted20[-1]) 
ogv_data_weighted25unwrap=np.unwrap(ogv_data_weighted25[-1]) 
ogv_data_weighted30unwrap=np.unwrap(ogv_data_weighted30[-1]) 
ogv_data_weighted60unwrap=np.unwrap(ogv_data_weighted60[-1]) 
ogv_data_weighted90unwrap=np.unwrap(ogv_data_weighted90[-1]) 
ogv_data_weighted100unwrap=np.unwrap(ogv_data_weighted100[-1])
ogv_data_weighted120unwrap=np.unwrap(ogv_data_weighted120[-1])

Min = -5.040e-12*2
Max = -5.345e-12*2
#cropping
d90, t90 = cropByTimestamp(ogv_data_weighted90unwrap,timestamps90, Max, Min)
print(timestamps90)
fit90 = np.poly1d(np.polyfit(t90,d90,1))
avgRad90 = np.nanmean(cropByTimestamp(ogv_data_weighted90[3],timestamps90,Max,Min))

d30, t30 = cropByTimestamp(ogv_data_weighted30unwrap,timestamps30,Max,Min)
fit30 = np.poly1d(np.polyfit(t30,d30,1))
avgRad30 = np.nanmean(cropByTimestamp(ogv_data_weighted30[3],timestamps30,Max,Min))

d20, t20 = cropByTimestamp(ogv_data_weighted20unwrap,timestamps20,Max,Min)
fit20 = np.poly1d(np.polyfit(t20,d20,1))
avgRad20 = np.nanmean(cropByTimestamp(ogv_data_weighted20[3],timestamps20,Max,Min))

d25, t25 = cropByTimestamp(ogv_data_weighted25unwrap,timestamps25,Max,Min)
fit25 = np.poly1d(np.polyfit(t25,d25,1))
avgRad25 = np.nanmean(cropByTimestamp(ogv_data_weighted25[3],timestamps25,Max,Min))

d60, t60 = cropByTimestamp(ogv_data_weighted60unwrap,timestamps60,Max,Min)
fit60 = np.poly1d(np.polyfit(t60,d60,1))
avgRad60 = np.nanmean(cropByTimestamp(ogv_data_weighted60[3],timestamps60,Max,Min))

d100, t100 = cropByTimestamp(ogv_data_weighted100unwrap,timestamps100,Max,Min)
fit100 = np.poly1d(np.polyfit(t100,d100,1))
avgRad100 = np.nanmean(cropByTimestamp(ogv_data_weighted100[3],timestamps100,Max,Min))

d120, t120 = cropByTimestamp(ogv_data_weighted120unwrap,timestamps120,Max,Min)
fit120 = np.poly1d(np.polyfit(t120,d120,1))
avgRad120 = np.nanmean(cropByTimestamp(ogv_data_weighted120[3],timestamps120,Max,Min))

plt.figure()
plt.plot(t20*1e15,d20, ls = ":", marker = "*", label = "angle weighted tc20", color = "r")
plt.plot(t20*1e15,fit20(t20), ls = "-", marker = "*", label = "Linear Fit tc20", color = "r")
plt.plot(t25*1e15,d25, ls = ":", marker="^", label = "angle weighted tc25", color = "orange")
plt.plot(t25*1e15,fit25(t25), ls = "-", marker="^", label = "Linear Fit tc25", color = "orange")
plt.plot(t30*1e15,d30, ls = ":", marker = "<", label = "angle weighted tc30", color = "y")
plt.plot(t30*1e15,fit30(t30), ls = "-",marker = "<", label = "Linear Fit tc30", color = "y" )
plt.plot(t60*1e15,d60, ls = ":", marker=".", label = "angle weighted tc60", color = "g")
plt.plot(t60*1e15,fit60(t60), ls = "-", marker=".", label = "Linear Fit tc60", color = "g")
plt.plot(t90*1e15,d90, ls = ":", marker = "o", label = "angle weighted tc90", color = "b")
plt.plot(t90*1e15,fit90(t90), ls = "-",marker = "o", label = "Linear Fit tc90", color = "b")
plt.plot(t100*1e15,d100, ls = ":", marker="v", label = "angle weighted tc100", color = "pink")
plt.plot(t100*1e15,fit100(t100), ls = "-", marker="v", label = "Linear Fit tc100", color = "pink")
plt.plot(t120*1e15,d120, ls = ":", marker="+", label = "angle weighted tc120", color = "purple")
plt.plot(t120*1e15,fit120(t120), ls = "-", marker="+", label = "Linear Fit tc120",color = "purple")
# plt.plot(timestamps*1e14,ogv_data_max[-1], label = "angle max")
plt.xlabel("time/fs")
plt.ylabel("angles/rad")
plt.legend()
plt.show(
)

OGV_theory = (2*np.pi*C0/778e-9 - 2*np.pi*C0/806e-9)/9

plt.figure()
#plt.plot((timestamps90*1e15),ogv_data_weighted90[2]/C0, label = "OGV_tc90 (dphi/dt)")
#plt.plot((timestamps30*1e15),ogv_data_weighted30[2]/C0, label = "OGV_tc30 (dphi/dt)")
plt.plot(t20*1e15,fit20.deriv(1)(t20), "--", label = "Linear Fit tc20", color = "r" )
plt.plot(t25*1e15,fit25.deriv(1)(t25), "--", label = "Linear Fit tc25",color = "orange" )
plt.plot(t30*1e15,fit30.deriv(1)(t30), "--", label = "Linear Fit tc30", color = "y" )
plt.plot(t60*1e15,fit60.deriv(1)(t60), "--", label = "Linear Fit tc60", color = "g" )
plt.plot(t90*1e15,fit90.deriv(1)(t90), "--", label = "Linear Fit tc90", color = "b")
plt.plot(t100*1e15,fit100.deriv(1)(t100), "--", label = "Linear Fit tc100", color = "pink" )
plt.plot(t120*1e15,fit120.deriv(1)(t120), "--", label = "Linear Fit tc120",color = "purple" )
plt.plot(t20*1e15, [-OGV_theory]*t20.size, "-", label = "Theoretical OGV")
plt.ylabel("OGV")
plt.xlabel("time/fs")
plt.legend()
plt.show()

theory90 = calculateLOGV_theory(avgRad90, dl= 9)/C0
theory30 = calculateLOGV_theory(avgRad30, dl= 9)/C0
theory20 = calculateLOGV_theory(avgRad20, dl= 9)/C0
theory25 = calculateLOGV_theory(avgRad25, dl= 9)/C0
theory60 = calculateLOGV_theory(avgRad60, dl= 9)/C0
theory100 = calculateLOGV_theory(avgRad100, dl= 9)/C0
theory120 = calculateLOGV_theory(avgRad120, dl= 9)/C0

plt.figure()
#plt.plot(timestamps90*1e15, ogv_data_weighted90[2]*ogv_data_weighted90[3]/C0, label = "LOGV 90")
#plt.plot(timestamps30*1e15, ogv_data_weighted30[2]*ogv_data_weighted30[3]/C0, label = "LOGV 30")
plt.plot([timestamps90.min()*1e15,timestamps90.max()*1e15],[avgRad90*fit90.deriv(1)(0)/C0]*2, ls = ":", c = "b", label = "LOGV tc90 Fit")
plt.plot([timestamps90.min()*1e15,timestamps90.max()*1e15],[-theory90]*2, ls = "-", label = "LOGV tc90 theory")
plt.plot([timestamps30.min()*1e15,timestamps30.max()*1e15],[-theory30]*2, ls = "-", label = "LOGV tc30 theory",color = "y")
plt.plot([timestamps30.min()*1e15,timestamps30.max()*1e15],[avgRad30*fit30.deriv(1)(0)/C0]*2, ls = ":",  label = "LOGV tc30 Fit",color = "y")

plt.plot([timestamps120.min()*1e15,timestamps120.max()*1e15],[-theory120]*2, ls = "-", label = "LOGV tc120 theory",color = "purple")
plt.plot([timestamps120.min()*1e15,timestamps120.max()*1e15],[avgRad120*fit120.deriv(1)(0)/C0]*2, ls = ":", label = "LOGV tc120 Fit",color = "purple")
plt.plot([timestamps100.min()*1e15,timestamps100.max()*1e15],[-theory100]*2, ls = "-", label = "LOGV tc100 theory",color = "pink")
plt.plot([timestamps100.min()*1e15,timestamps100.max()*1e15],[avgRad100*fit100.deriv(1)(0)/C0]*2, ls = ":", label = "LOGV tc100 Fit",color = "pink")
plt.plot([timestamps60.min()*1e15,timestamps60.max()*1e15],[-theory60]*2, ls = "-", label = "LOGV tc60 theory",color = "g")
plt.plot([timestamps60.min()*1e15,timestamps60.max()*1e15],[avgRad60*fit60.deriv(1)(0)/C0]*2, ls = ":", label = "LOGV tc60 Fit",color = "g" )
plt.plot([timestamps25.min()*1e15,timestamps25.max()*1e15],[-theory25]*2, ls = "-", label = "LOGV tc25 theory",color = "orange" )
plt.plot([timestamps25.min()*1e15,timestamps25.max()*1e15],[avgRad25*fit25.deriv(1)(0)/C0]*2, ls = ":", label = "LOGV tc25 Fit",color = "orange")
plt.plot([timestamps20.min()*1e15,timestamps20.max()*1e15],[-theory20]*2, ls = "-", label = "LOGV tc20 theory",color = "r")
plt.plot([timestamps20.min()*1e15,timestamps20.max()*1e15],[avgRad20*fit20.deriv(1)(0)/C0]*2, ls = ":", label = "LOGV tc20 Fit",color = "r")

plt.title("LOGV")
plt.ylabel("LOGV/ c")
plt.xlabel("time/fs")
plt.legend()
plt.show()

plt.figure()
plt.plot([timestamps20.min()*1e15,timestamps20.max()*1e15],[avgRad20]*2, ls = ":", label = "Radius tc20 Fit" )
plt.plot([timestamps25.min()*1e15,timestamps25.max()*1e15],[avgRad25]*2, ls = ":",label = "Radius tc25 Fit" )
plt.plot([timestamps30.min()*1e15,timestamps30.max()*1e15],[avgRad30]*2, ls = ":",  label = "Radius tc30 Fit" )
plt.plot([timestamps60.min()*1e15,timestamps60.max()*1e15],[avgRad60]*2, ls = ":",  label = "Radius tc60 Fit" )
plt.plot([timestamps90.min()*1e15,timestamps90.max()*1e15],[avgRad90]*2, ls = ":",  label = "Radius tc90 Fit" )
plt.plot([timestamps100.min()*1e15,timestamps100.max()*1e15],[avgRad100]*2, ls = ":",  label = "Radius tc100 Fit" )
plt.plot([timestamps120.min()*1e15,timestamps120.max()*1e15],[avgRad120]*2, ls = ":",  label = "Radius tc120 Fit" )
plt.ylabel("Radius/ m")
plt.xlabel("time/fs")
plt.legend()
plt.show()

#print(avgRad90)


In [ ]:
print(avgRad30)

In [ ]:
#export cmaps
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

path = Path(r"C:\Users\Voxel\Desktop\LightSprings")
cmap = "magma"

plasma = plt.get_cmap(cmap, 256)
colors = plasma(np.linspace(0, 1, 256))[:, :3]
cmap = cmap+".txt"

np.savetxt(path/cmap, colors)